# [재학습 · 2026-10-05 회계 수정] 4.3절 목적함수 주입 실험

원본 노트북 §21(v3 + β-조건부 현금 보조보상)을 **회계 수정 후 환경**(보유 비중 drift, 거래 불가 종목 마스크)에서 처음부터 다시 학습한다.
- v3 baseline 풀(보조보상 없음), λ=0.01 풀, λ=0.15 풀 — 각 4β × 100k 스텝 = PPO 12개
- 다양성 진단(β별 평균 현금비중, β 순서와의 스피어만 상관)과 α=0.01 스무딩 valid/test 성과

기존 모델을 덮어쓰지 않도록 모델 폴더와 CSV에 `_fix1004`를 붙였다. 위에서부터 **순서대로 전부 실행**(CPU, 수 시간).


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q stable-baselines3[extra] gymnasium h5py pyarrow


In [ ]:
# 공통 설정(경로·하이퍼파라미터, 임베딩·수익률 로드, 회계 수정 환경, 버킷 분할)은
# 같은 폴더의 crypto_ddqn_retrain_fix1004.ipynb 2~6번 셀과 동일하다 — 그 셀들을 그대로 불러와 실행한다.
import json as _json
from pathlib import Path as _Path
_shared = _Path('/content/drive/MyDrive/졸업프로젝트/rerun_fix1004b/crypto_ddqn_retrain_fix1004.ipynb')
assert _shared.exists(), f'공통 셀 원본 없음: {_shared}'
_cells = _json.load(open(_shared))['cells']
for _c in _cells[2:7]:
    _s = ''.join(_c['source']) if isinstance(_c['source'], list) else _c['source']
    assert '_drift_weights' in _s or 'DRIVE_ROOT' in _s or 'h5py.File' in _s or 'RetTarget_6b' in _s or 'bucket_to_ts' in _s
    exec(_s, globals())
assert '_drift_weights' in globals() and 'env_kwargs' in globals()
print('공통 셀 실행 완료 (회계 수정 환경 포함)')


In [ ]:
# crypto/market_segmentation.py 이식 (Colab은 저장소를 마운트하지 않으므로 동일 코드를 셀에 포함)
REGIME_NAMES = {1: 'bear', 2: 'pullback', 3: 'sideways', 4: 'rally', 5: 'bull'}
_DTW_MAX_LEN = 200


@dataclass
class MarketSegments:
    labels: pd.Series
    chunks: pd.DataFrame


def _zscore(x):
    std = x.std()
    return (x - x.mean()) / std if std > 1e-12 else np.zeros_like(x)


def _downsample(x, max_len=_DTW_MAX_LEN):
    if len(x) <= max_len:
        return x
    idx = np.linspace(0, len(x) - 1, max_len).round().astype(int)
    return x[idx]


def _dtw_distance(a, b):
    a, b = _downsample(a), _downsample(b)
    n, m = len(a), len(b)
    cost = np.full((n + 1, m + 1), np.inf)
    cost[0, 0] = 0.0
    for i in range(1, n + 1):
        ai = a[i - 1]
        row, prev = cost[i], cost[i - 1]
        for j in range(1, m + 1):
            row[j] = abs(ai - b[j - 1]) + min(prev[j], row[j - 1], prev[j - 1])
    return float(cost[n, m])


def _slope(x):
    if len(x) < 2:
        return 0.0
    t = np.arange(len(x), dtype=np.float64)
    a, _ = np.polyfit(t, x, 1)
    return float(a)


def _find_extrema_indices(x):
    d = np.diff(x)
    sign = np.sign(d)
    sign[sign == 0] = 1
    change = np.where(np.diff(sign) != 0)[0] + 1
    idx = np.concatenate(([0], change, [len(x) - 1]))
    return np.unique(idx)


def _merge_pass(x, bounds, merge_quantile):
    if len(bounds) <= 2:
        return bounds, False
    seg_ranges = list(zip(bounds[:-1], bounds[1:]))
    slopes = [_slope(x[a:b + 1]) for a, b in seg_ranges]
    if len(seg_ranges) < 2:
        return bounds, False
    slope_diffs = [abs(slopes[i + 1] - slopes[i]) for i in range(len(slopes) - 1)]
    dtw_dists = [
        _dtw_distance(
            _zscore(x[seg_ranges[i][0]:seg_ranges[i][1] + 1]),
            _zscore(x[seg_ranges[i + 1][0]:seg_ranges[i + 1][1] + 1]),
        )
        for i in range(len(seg_ranges) - 1)
    ]
    slope_thresh = np.quantile(slope_diffs, merge_quantile)
    dtw_thresh = np.quantile(dtw_dists, merge_quantile)

    new_bounds = [bounds[0]]
    merged_any = False
    i = 0
    while i < len(seg_ranges):
        can_merge = (
            i < len(seg_ranges) - 1
            and slope_diffs[i] <= slope_thresh
            and dtw_dists[i] <= dtw_thresh
        )
        if can_merge:
            new_bounds.append(seg_ranges[i + 1][1])
            merged_any = True
            i += 2
        else:
            new_bounds.append(seg_ranges[i][1])
            i += 1
    return new_bounds, merged_any


def _absorb_short_segments(x, bounds, min_length):
    while len(bounds) > 2:
        seg_ranges = list(zip(bounds[:-1], bounds[1:]))
        lengths = [b - a + 1 for a, b in seg_ranges]
        shortest = int(np.argmin(lengths))
        if lengths[shortest] >= min_length:
            break
        slopes = [_slope(x[a:b + 1]) for a, b in seg_ranges]
        left_diff = abs(slopes[shortest] - slopes[shortest - 1]) if shortest > 0 else np.inf
        right_diff = (
            abs(slopes[shortest] - slopes[shortest + 1]) if shortest < len(seg_ranges) - 1 else np.inf
        )
        drop_idx = shortest if left_diff <= right_diff else shortest + 1
        bounds = bounds[:drop_idx] + bounds[drop_idx + 1:]
    return bounds


def segment_and_label(series, n_labels=5, risk_threshold=0.2, lowpass_window=48,
                       merge_quantile=0.35, max_merge_iters=50, min_segment_length=None):
    """가격 시계열을 레짐 구간으로 분할하고 라벨을 부여한다 (EarnHFT Algorithm 3 이식).
    로그가격(buy&hold net curve의 로그) 기준으로 계산해 가격 스케일 왜곡을 방지한다."""
    x_raw = series.to_numpy(dtype=np.float64)
    if len(x_raw) < 3:
        raise ValueError('series가 너무 짧습니다 (최소 3개 포인트 필요)')
    if np.any(x_raw <= 0):
        raise ValueError('series는 로그 변환을 위해 모두 양수여야 합니다')

    x_log = np.log(x_raw)
    x_smooth = pd.Series(x_log).rolling(lowpass_window, min_periods=1, center=True).mean().to_numpy()

    bounds = _find_extrema_indices(x_smooth).tolist()
    for _ in range(max_merge_iters):
        bounds, merged = _merge_pass(x_smooth, bounds, merge_quantile)
        if not merged or len(bounds) <= 2:
            break
    bounds = _absorb_short_segments(x_log, bounds, min_segment_length or lowpass_window)

    seg_ranges = list(zip(bounds[:-1], bounds[1:]))
    slopes = np.array([_slope(x_log[a:b + 1]) for a, b in seg_ranges])

    theta = risk_threshold
    H = np.quantile(slopes, 1 - theta / 2)
    L = np.quantile(slopes, theta / 2)

    labels = np.empty(len(slopes), dtype=np.int64)
    for i, s in enumerate(slopes):
        if s > H:
            labels[i] = n_labels
        elif s < L:
            labels[i] = 1
        elif H > L:
            frac = (s - L) / (H - L)
            labels[i] = int(np.clip(2 + frac * (n_labels - 2), 2, n_labels - 1))
        else:
            labels[i] = (n_labels + 1) // 2

    per_point_labels = np.empty(len(x_raw), dtype=np.int64)
    for (a, b), lbl in zip(seg_ranges, labels):
        per_point_labels[a:b + 1] = lbl

    labels_series = pd.Series(per_point_labels, index=series.index, name='regime_label')

    chunks = pd.DataFrame({
        'start_idx': [a for a, _ in seg_ranges],
        'end_idx': [b for _, b in seg_ranges],
        'length': [b - a + 1 for a, b in seg_ranges],
        'slope': slopes,
        'label': labels,
    })
    chunks['regime'] = chunks['label'].map(REGIME_NAMES) if n_labels == 5 else chunks['label'].astype(str)
    return MarketSegments(labels=labels_series, chunks=chunks)


def chunk_sampling_priority(chunks, beta, risk_threshold=0.2):
    """EarnHFT Eq.4 이식 — chunk 기울기(r)에 대한 β-지수가중 샘플링 우선순위(합 1)."""
    r = chunks['slope'].to_numpy(dtype=np.float64)
    theta = risk_threshold
    Ht = np.quantile(r, 1 - theta / 2)
    Lt = np.quantile(r, theta / 2)

    std = r.std() if r.std() > 1e-12 else 1.0
    h = max(0.9 * std * len(r) ** (-1 / 5), 1e-8)
    diffs = (r[:, None] - r[None, :]) / h
    pdf = np.exp(-0.5 * diffs ** 2).sum(axis=1) / (len(r) * h * np.sqrt(2 * np.pi))
    pdf = np.maximum(pdf, 1e-12)

    tail_mask = (r >= Ht) | (r <= Lt)
    log_w = beta * r - np.where(tail_mask, 0.0, np.log(pdf))
    log_w -= log_w.max()
    weights = np.exp(log_w)
    return weights / weights.sum()


print('market_segmentation 함수 정의 완료')


In [ ]:
# §12-A — 현금 액션 환경 + 2025 내부 train/valid 분할 + v3 라우터 환경 정의
V3_TRAIN_END   = '2025-09-30'   # Stage B 학습: 2025-01 ~ 2025-09
V3_VALID_START = '2025-10-01'   # 선별/검증: 2025-10 ~ 2025-12 (테스트 2026은 최종 1회만)
V3_POOL_DIR    = MODEL_DIR / 'crypto_hrl_pool_v3'
V3_INFER_ALPHA = 0.03           # 선별 단계 실행 스무딩 기본값 — 최종 α는 §12-F에서 valid로 선택

v3_train_buckets = [b for b in train_buckets if bucket_to_date(b) <= V3_TRAIN_END]
v3_valid_buckets = [b for b in train_buckets if bucket_to_date(b) >= V3_VALID_START]
print(f'v3 학습: {bucket_to_date(v3_train_buckets[0])} ~ {bucket_to_date(v3_train_buckets[-1])} ({len(v3_train_buckets):,}스텝)')
print(f'v3 검증: {bucket_to_date(v3_valid_buckets[0])} ~ {bucket_to_date(v3_valid_buckets[-1])} ({len(v3_valid_buckets):,}스텝)')

if 'ew_ret_by_bucket' not in globals():
    ew_ret_by_bucket = {
        b: float(np.mean([ret_index.get((s, b), 0.0) for s in symbols]))
        for b in all_buckets
    }


class CashPortfolioEnv(CryptoPortfolioEnv):
    """44종목 + 현금(마지막 인덱스, 수익률 0) — 논문의 'position 0'(청산) 대응.
    β<0 에이전트가 하락장 대응(포지션 축소)을 액션으로 직접 표현할 수 있게 한다.
    거래비용은 자산 레그 턴오버에만 부과(현금 이동 자체는 무료)."""

    def __init__(self, buckets, **kwargs):
        super().__init__(buckets, **kwargs)
        self.action_space = spaces.Box(-3.0, 3.0, shape=(self.n_assets + 1,), dtype=np.float32)
        self.observation_space = spaces.Box(
            -np.inf, np.inf, shape=(self.n_assets * self.emb_dim + self.n_assets + 1,), dtype=np.float32)

    def reset(self, seed=None, options=None):
        gym.Env.reset(self, seed=seed)
        self.t                = 0
        self.weights          = _mask_untradable(np.ones(self.n_assets + 1, dtype=np.float32),
                                          np.append(self._tradable(self.buckets[0]), True))  # 거래 가능 종목+현금 등가중에서 시작
        self.portfolio_value  = 1.0
        self.value_history    = [1.0]
        self.turnover_history = []
        return self._get_obs(), {}

    def step(self, action):
        w = np.exp(action - action.max())
        w = w / w.sum()                                                   # 45-dim, 합=1
        w = _mask_untradable(w, np.append(self._tradable(self.buckets[self.t]), True))  # 거래 불가 종목 제외
        asset_turnover = float(np.abs(w[:-1] - self.weights[:-1]).sum())  # 자산 레그만
        tc_cost = self.tc * asset_turnover

        b        = self.buckets[self.t]
        port_ret = float((w[:-1] * self._asset_ret(b)).sum())             # 현금 수익률 0

        self.portfolio_value *= (1 + port_ret - tc_cost)
        self.weights = _drift_weights(w, np.append(self._asset_ret(b), 0.0))  # 가격 변동 후 실제 보유 비중
        self.value_history.append(self.portfolio_value)
        self.turnover_history.append(asset_turnover)
        self.t += 1

        done = (self.t >= len(self.buckets) - 1)
        info = {'portfolio_value': self.portfolio_value, 'port_ret': port_ret,
                'turnover': asset_turnover, 'tc_cost': tc_cost, 'cash': float(w[-1])}
        return self._get_obs(), float(port_ret - tc_cost), done, False, info


class ChunkedCashEnv(CashPortfolioEnv):
    """β 우선순위 청크 샘플링 에피소드 — §5 ChunkedCryptoPortfolioEnv와 동일 로직, 환경만 현금 버전."""

    def __init__(self, all_buckets, chunks, beta, risk_threshold=0.2, rng=None, **env_kwargs):
        self.all_buckets = all_buckets
        self.chunks      = chunks.reset_index(drop=True)
        self.beta        = beta
        self.priority    = chunk_sampling_priority(self.chunks, beta, risk_threshold)
        self.rng         = rng or np.random.default_rng()
        super().__init__(buckets=all_buckets[:2], **env_kwargs)

    def reset(self, seed=None, options=None):
        idx = self.rng.choice(len(self.chunks), p=self.priority)
        row = self.chunks.iloc[idx]
        self.buckets = self.all_buckets[int(row['start_idx']):int(row['end_idx']) + 1]
        return super().reset(seed=seed, options=options)


check_env(CashPortfolioEnv(v3_train_buckets[:100], **env_kwargs), warn=True)
print('CashPortfolioEnv 유효성 OK (obs dim =', 44 * EMB_DIM + 45, ')')


In [ ]:
# §12-B — Stage A 재실행: v3 학습구간 청크(β 샘플링용) + 검증구간 라벨(에이전트 선별용)
def ew_nav_of(buckets):
    r = np.array([ew_ret_by_bucket.get(b, 0.0) for b in buckets])
    return pd.Series(np.cumprod(1 + r), index=range(len(buckets)))

seg_v3_train = segment_and_label(
    ew_nav_of(v3_train_buckets), n_labels=N_LABELS,
    risk_threshold=RISK_THRESHOLD, lowpass_window=LOWPASS_WINDOW,
)
seg_v3_valid = segment_and_label(
    ew_nav_of(v3_valid_buckets), n_labels=N_LABELS,
    risk_threshold=RISK_THRESHOLD, lowpass_window=LOWPASS_WINDOW,
)
print(f"학습 청크: {len(seg_v3_train.chunks)}개 | 레짐 분포: {seg_v3_train.chunks['regime'].value_counts().to_dict()}")
print(f"검증 청크: {len(seg_v3_valid.chunks)}개 | 레짐 분포: {seg_v3_valid.chunks['regime'].value_counts().to_dict()}")


In [ ]:
# §13-K-A — valid 구간 정의 (v3 §12와 동일 분할 재사용: train 1~9월 / valid 10~12월)
VALID_START, VALID_END = '2025-10-01', '2025-12-31'
valid_buckets = [b for b in train_buckets if VALID_START <= bucket_to_date(b) <= VALID_END]
print(f'valid 구간: {bucket_to_ts(valid_buckets[0])} ~ {bucket_to_ts(valid_buckets[-1])} ({len(valid_buckets):,}스텝)')


In [ ]:
def perf_num(values, steps_per_year=48 * 365):
    """수치형 성과 — 논문 표에 바로 옮길 수 있게 float로 반환"""
    values = np.asarray(values, dtype=np.float64)
    rets = np.diff(values) / values[:-1]
    return {'ret': values[-1] / values[0] - 1,
            'sharpe': rets.mean() / (rets.std() + 1e-8) * np.sqrt(steps_per_year),
            'mdd': ((values / np.maximum.accumulate(values)) - 1).min()}

compute_perf = perf_num   # 원본 셀들이 부르는 이름 유지

AUX_LAMBDA = 0.01
AUX_LAMBDA_STRONG = 0.15


In [ ]:
class ObjectiveInjectedCashEnv(ChunkedCashEnv):
    """
    v3(§12)에서 실제로 진단된 붕괴(현금 비중이 전 구간 1/45≈2.2%로 고정)를 정확히 겨냥한 목적함수
    주입. ChunkedCashEnv(§12-A)와 완전히 동일하되(Stage A 청크 샘플링·β 우선순위 로직 그대로
    재사용), step()에 β-조건부 보조보상 항만 추가한다.

    보조보상: aux = -AUX_LAMBDA * beta_norm * (cash - 0.5)
      - cash = 이번 스텝에서 실제로 보유하게 된 현금 비중(w[-1])
      - beta_norm = beta / 90 ∈ {-1, -1/3, 1/3, 1}
      - β>0(공격적)이면 현금 적게 들수록 보너스, β<0(방어적)이면 현금 많이 들수록 보너스 —
        v3가 실제로 학습하지 못했던 바로 그 구분(β에 따라 현금 비중을 다르게 가져가는 것)을
        목적함수에 직접 심는다. cash=0.5를 중심으로 대칭 설계(0이면 전액 투자, 1이면 전액 현금).
    """

    def __init__(self, *args, aux_lambda=0.01, **kwargs):
        self.aux_lambda = aux_lambda
        super().__init__(*args, **kwargs)

    def step(self, action):
        obs, reward, done, truncated, info = super().step(action)
        cash = float(self.weights[-1])
        beta_norm = self.beta / 90.0
        aux_reward = -self.aux_lambda * beta_norm * (cash - 0.5)
        reward = reward + aux_reward
        info['aux_reward'] = aux_reward
        return obs, reward, done, truncated, info


# 스모크 테스트 — β 부호대로 aux_reward 부호가 나오는지 확인
if 'seg_v3_train' not in globals():
    # §12-B 미실행 세션에서도 21-A 스모크 테스트는 돌아가도록, 검증용 더미 청크 하나만 구성
    _DUMMY_CHUNKS = pd.DataFrame([
        {'start_idx': 0, 'end_idx': min(200, len(v3_train_buckets) - 1), 'length': 200,
         'slope': 0.0, 'regime': 'sideways'},
    ])
    _smoke_chunks = _DUMMY_CHUNKS
    print('[§21-A] seg_v3_train 없음 — 스모크 테스트용 더미 청크로 대체(§12-B 실행 시 자동으로 안 씀)')
else:
    _smoke_chunks = seg_v3_train.chunks

_smoke_cash_pos = ObjectiveInjectedCashEnv(
    all_buckets=v3_train_buckets, chunks=_smoke_chunks, beta=90, aux_lambda=AUX_LAMBDA,
    risk_threshold=RISK_THRESHOLD, rng=np.random.default_rng(0), **env_kwargs,
)
_smoke_cash_pos.reset()
_cash_heavy_action = np.full(N_ASSETS + 1, -3.0, dtype=np.float32)
_cash_heavy_action[-1] = 3.0  # 현금에 몰빵
_, _, _, _, _info_cash_pos = _smoke_cash_pos.step(_cash_heavy_action)

_smoke_cash_neg = ObjectiveInjectedCashEnv(
    all_buckets=v3_train_buckets, chunks=_smoke_chunks, beta=-90, aux_lambda=AUX_LAMBDA,
    risk_threshold=RISK_THRESHOLD, rng=np.random.default_rng(0), **env_kwargs,
)
_smoke_cash_neg.reset()
_, _, _, _, _info_cash_neg = _smoke_cash_neg.step(_cash_heavy_action)

print(f'β=90,  현금 몰빵 → cash={_smoke_cash_pos.weights[-1]:.3f}, aux_reward={_info_cash_pos["aux_reward"]:+.5f} (음수여야 함 — 공격적인데 현금만 들면 페널티)')
print(f'β=-90, 현금 몰빵 → cash={_smoke_cash_neg.weights[-1]:.3f}, aux_reward={_info_cash_neg["aux_reward"]:+.5f} (양수여야 함 — 방어적인데 현금 들면 보너스)')
assert _info_cash_pos['aux_reward'] < 0 and _info_cash_neg['aux_reward'] > 0, '보조보상 부호가 β 방향과 안 맞음 — 구현 확인 필요'
print('[OK] 보조보상 부호 검증 통과')

In [ ]:
%%time
# v3 baseline(보조보상 없음)도 수정된 환경에서 새로 학습 — 기존 체크포인트는 수정 전 보상으로 학습된 것이라 쓰지 않는다
V3_BASE_DIR_FIX = MODEL_DIR / 'crypto_hrl_pool_v3_base_fix1004'
V3_BASE_DIR_FIX.mkdir(parents=True, exist_ok=True)
v3_baseline_models = {}
for pool_i, beta in enumerate(POOL_BETAS):
    print(f'\n=== [v3 baseline] β={beta} 학습 ({POOL_TIMESTEPS_EACH:,} 스텝) ===')
    rng = np.random.default_rng(4000 + pool_i)
    chunk_env = Monitor(ChunkedCashEnv(
        all_buckets=v3_train_buckets, chunks=seg_v3_train.chunks, beta=beta,
        risk_threshold=RISK_THRESHOLD, rng=rng, **env_kwargs,
    ))
    model = PPO(
        'MlpPolicy', chunk_env,
        learning_rate=3e-4, n_steps=1024, batch_size=64, n_epochs=10,
        gamma=0.99, gae_lambda=0.95, clip_range=0.2, ent_coef=0.01, vf_coef=0.5,
        max_grad_norm=0.5, policy_kwargs=dict(net_arch=[512, 256]),
        verbose=0, device='cpu', seed=42,
    )
    model.learn(total_timesteps=POOL_TIMESTEPS_EACH, progress_bar=True)
    model.save(str(V3_BASE_DIR_FIX / f'agent_beta_{beta}.zip'))
    v3_baseline_models[beta] = model
print(f'\nv3 baseline 풀 학습 완료: {len(v3_baseline_models)}개')


In [ ]:
%%time
POOL_DIR_OBJINJ_CASH = MODEL_DIR / 'crypto_hrl_pool_objinj_cash_fix1004'
POOL_DIR_OBJINJ_CASH.mkdir(parents=True, exist_ok=True)
pool_models_objinj_cash = {}

assert 'seg_v3_train' in globals(), '§12-B(Stage A 재분할)를 먼저 실행하세요 — seg_v3_train 필요'

for pool_i, beta in enumerate(POOL_BETAS):
    print(f'\n=== [v3 + 목적함수 주입] β={beta} 에이전트 학습 ({POOL_TIMESTEPS_EACH:,} 스텝) ===')
    rng = np.random.default_rng(5000 + pool_i)  # 다른 풀들과 겹치지 않는 시드 대역
    chunk_env = Monitor(ObjectiveInjectedCashEnv(
        all_buckets=v3_train_buckets, chunks=seg_v3_train.chunks, beta=beta,
        risk_threshold=RISK_THRESHOLD, aux_lambda=AUX_LAMBDA, rng=rng, **env_kwargs,
    ))
    model = PPO(
        'MlpPolicy', chunk_env,
        learning_rate=3e-4, n_steps=1024, batch_size=64, n_epochs=10,
        gamma=0.99, gae_lambda=0.95, clip_range=0.2, ent_coef=0.01, vf_coef=0.5,
        max_grad_norm=0.5, policy_kwargs=dict(net_arch=[512, 256]),
        verbose=0, device='cpu', seed=42,
    )
    model.learn(total_timesteps=POOL_TIMESTEPS_EACH, progress_bar=True)

    save_path = POOL_DIR_OBJINJ_CASH / f'agent_beta_{beta}.zip'
    model.save(str(save_path))
    pool_models_objinj_cash[beta] = model
    print(f'저장: {save_path.name}')

print(f'\n[v3 + 목적함수 주입] 풀 학습 완료: {len(pool_models_objinj_cash)}개 에이전트')

In [ ]:
%%time
POOL_DIR_OBJINJ_CASH_STRONG = MODEL_DIR / 'crypto_hrl_pool_objinj_cash_strong_fix1004'
POOL_DIR_OBJINJ_CASH_STRONG.mkdir(parents=True, exist_ok=True)
pool_models_objinj_cash_strong = {}

assert 'seg_v3_train' in globals(), '§12-B(Stage A 재분할)를 먼저 실행하세요 — seg_v3_train 필요'

for pool_i, beta in enumerate(POOL_BETAS):
    print(f'\n=== [v3 + 목적함수 주입 STRONG] β={beta} 에이전트 학습 ({POOL_TIMESTEPS_EACH:,} 스텝) ===')
    rng = np.random.default_rng(6000 + pool_i)  # 다른 풀들과 겹치지 않는 시드 대역
    chunk_env = Monitor(ObjectiveInjectedCashEnv(
        all_buckets=v3_train_buckets, chunks=seg_v3_train.chunks, beta=beta,
        risk_threshold=RISK_THRESHOLD, aux_lambda=AUX_LAMBDA_STRONG, rng=rng, **env_kwargs,
    ))
    model = PPO(
        'MlpPolicy', chunk_env,
        learning_rate=3e-4, n_steps=1024, batch_size=64, n_epochs=10,
        gamma=0.99, gae_lambda=0.95, clip_range=0.2, ent_coef=0.01, vf_coef=0.5,
        max_grad_norm=0.5, policy_kwargs=dict(net_arch=[512, 256]),
        verbose=0, device='cpu', seed=42,
    )
    model.learn(total_timesteps=POOL_TIMESTEPS_EACH, progress_bar=True)

    save_path = POOL_DIR_OBJINJ_CASH_STRONG / f'agent_beta_{beta}.zip'
    model.save(str(save_path))
    pool_models_objinj_cash_strong[beta] = model
    print(f'저장: {save_path.name}')

print(f'\n[v3 + 목적함수 주입 STRONG] 풀 학습 완료: {len(pool_models_objinj_cash_strong)}개 에이전트')

In [ ]:
def run_backtest_smoothed(env_cls, model, buckets, alpha=0.01):
    """target = softmax(model 예측 로짓), 실행은 관성 스무딩(w_new=(1-α)w_old+α·target)으로 적용."""
    env = env_cls(buckets, **env_kwargs)
    obs, _ = env.reset()
    done, values = False, [1.0]
    while not done:
        a, _ = model.predict(obs, deterministic=True)
        target = np.exp(a - a.max())
        target = target / target.sum()
        w = (1 - alpha) * env.weights + alpha * target
        obs, _, done, _, info = env.step(np.log(np.maximum(w, 1e-12)))
        values.append(info['portfolio_value'])
    return np.array(values), list(env.turnover_history)


In [ ]:
def collect_cash_trajectory(model, buckets):
    """결정론적 실행으로 스텝별 실제 비중(45차원, 현금 포함) 궤적을 수집."""
    env = CashPortfolioEnv(buckets, **env_kwargs)
    obs, _ = env.reset()
    done, weights_hist, cash_hist = False, [], []
    while not done:
        action, _ = model.predict(obs, deterministic=True)
        obs, _, done, _, info = env.step(action)
        weights_hist.append(env.weights.copy())
        cash_hist.append(float(env.weights[-1]))
    return np.array(weights_hist), np.array(cash_hist)


def pairwise_mean_corr_v2(weight_trajs):
    """β쌍마다 전 차원(자산+현금) 시계열 상관계수를 구해 평균 — 1에 가까울수록 다양성 없음."""
    betas = list(weight_trajs.keys())
    corrs = {}
    for i in range(len(betas)):
        for j in range(i + 1, len(betas)):
            a, b = weight_trajs[betas[i]], weight_trajs[betas[j]]
            n = min(len(a), len(b))
            dims = a.shape[1]
            per_dim_corr = [
                np.corrcoef(a[:n, k], b[:n, k])[0, 1]
                for k in range(dims)
                if a[:n, k].std() > 1e-8 and b[:n, k].std() > 1e-8
            ]
            corrs[f'{betas[i]}\u2194{betas[j]}'] = float(np.mean(per_dim_corr)) if per_dim_corr else float('nan')
    return corrs


print('=== [§21] 다양성 진단: v3 baseline vs 목적함수 주입 (현금 비중 기준) ===\n')

cash_rows = {}
weight_trajs_v3_base, weight_trajs_v3_objinj = {}, {}

for beta in POOL_BETAS:
    w_traj, cash = collect_cash_trajectory(v3_baseline_models[beta], test_buckets)
    weight_trajs_v3_base[beta] = w_traj
    cash_rows[f'β={beta} (v3 baseline)'] = {'평균 현금비중': cash.mean(), '표준편차': cash.std()}

for beta in POOL_BETAS:
    w_traj, cash = collect_cash_trajectory(pool_models_objinj_cash[beta], test_buckets)
    weight_trajs_v3_objinj[beta] = w_traj
    cash_rows[f'β={beta} (목적함수 주입)'] = {'평균 현금비중': cash.mean(), '표준편차': cash.std()}

cash_df = pd.DataFrame(cash_rows).T
print('--- 현금 비중, β별 (test 구간, 1/45≈2.2%가 v3의 실제 붕괴값) ---')
print(cash_df.to_string(float_format=lambda x: f'{x:.4f}'))

corr_v3_base   = pairwise_mean_corr_v2(weight_trajs_v3_base)
corr_v3_objinj = pairwise_mean_corr_v2(weight_trajs_v3_objinj)
corr_df2 = pd.DataFrame({'목적함수 주입': corr_v3_objinj, 'v3 baseline': corr_v3_base})
print('\n--- β쌍 포트폴리오 비중 상관관계 (45차원, 현금 포함, 테스트 구간) ---')
print(corr_df2.to_string(float_format=lambda x: f'{x:.3f}'))
print('\n(v3 baseline은 현금 비중이 β 무관하게 거의 고정일 것으로 예상 — 목적함수 주입 쪽이 β별로 뚜렷이 갈리면 성공)')

cash_df.to_csv(CRYPTO_DIR / 'crypto_hrl_v3_objinj_diversity_cash_fix1004.csv', encoding='utf-8-sig')
corr_df2.to_csv(CRYPTO_DIR / 'crypto_hrl_v3_objinj_diversity_corr_fix1004.csv', encoding='utf-8-sig')
print('\n저장 완료: crypto_hrl_v3_objinj_diversity_cash_fix1004.csv, crypto_hrl_v3_objinj_diversity_corr_fix1004.csv')

In [ ]:
print('=== [§21-H] 3자 비교: v3 baseline vs 목적함수 주입(λ=0.01) vs 목적함수 주입(λ=0.15) ===\n')

cash_rows_3way = {}
cash_means = {'v3 baseline': {}, 'λ=0.01': {}, 'λ=0.15': {}}

for beta in POOL_BETAS:
    _, cash = collect_cash_trajectory(v3_baseline_models[beta], test_buckets)
    cash_rows_3way[f'β={beta} (v3 baseline)'] = {'평균 현금비중': cash.mean(), '표준편차': cash.std()}
    cash_means['v3 baseline'][beta] = cash.mean()

for beta in POOL_BETAS:
    _, cash = collect_cash_trajectory(pool_models_objinj_cash[beta], test_buckets)
    cash_rows_3way[f'β={beta} (λ=0.01)'] = {'평균 현금비중': cash.mean(), '표준편차': cash.std()}
    cash_means['λ=0.01'][beta] = cash.mean()

for beta in POOL_BETAS:
    _, cash = collect_cash_trajectory(pool_models_objinj_cash_strong[beta], test_buckets)
    cash_rows_3way[f'β={beta} (λ=0.15)'] = {'평균 현금비중': cash.mean(), '표준편차': cash.std()}
    cash_means['λ=0.15'][beta] = cash.mean()

cash_df_3way = pd.DataFrame(cash_rows_3way).T
print('--- 현금 비중, β별 (test 구간) ---')
print(cash_df_3way.to_string(float_format=lambda x: f'{x:.4f}'))

print('\n--- 몬오토닉성: β 순서 vs 현금비중 순서의 스피어만 상관 ---')
print('(설계 의도: β가 클수록/공격적일수록 현금비중이 낮아야 하므로 -1에 가까울수록 의도대로 작동)')
for label in ['v3 baseline', 'λ=0.01', 'λ=0.15']:
    betas_sorted = sorted(cash_means[label].keys())
    cash_vals = [cash_means[label][b] for b in betas_sorted]
    rho = pd.Series(betas_sorted).corr(pd.Series(cash_vals), method='spearman')
    print(f'  {label}: 스피어만 ρ(β, 현금비중) = {rho:+.3f}')

fig, ax = plt.subplots(figsize=(7, 4))
for label, marker in [('v3 baseline', 'o'), ('λ=0.01', 's'), ('λ=0.15', '^')]:
    betas_sorted = sorted(cash_means[label].keys())
    cash_vals = [cash_means[label][b] for b in betas_sorted]
    ax.plot(betas_sorted, cash_vals, marker=marker, label=label)
ax.axhline(1 / (N_ASSETS + 1), color='gray', ls=':', lw=1, label='1/45 (v3 원래 붕괴값)')
ax.set_xlabel('β (리스크 선호도)')
ax.set_ylabel('평균 현금비중 (test 구간)')
ax.set_title('β별 현금비중 — λ 강도에 따른 변화')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(CRYPTO_DIR / 'crypto_hrl_v3_objinj_lambda_sweep_fix1004.png', dpi=150, bbox_inches='tight')
plt.show()

cash_df_3way.to_csv(CRYPTO_DIR / 'crypto_hrl_v3_objinj_diversity_cash_3way_fix1004.csv', encoding='utf-8-sig')
print('\n저장 완료: crypto_hrl_v3_objinj_diversity_cash_3way_fix1004.csv, crypto_hrl_v3_objinj_lambda_sweep_fix1004.png')

In [ ]:
print('=== [§21-E] valid(2025 Q4)/test(2026~) 성과 — 스무딩(α=0.01) 적용, 라우터 없이 β 단독 ===\n')

def equal_weight_cash_backtest(buckets):
    env = CashPortfolioEnv(buckets, **env_kwargs)
    obs, _ = env.reset()
    target = np.zeros(env.n_assets + 1, dtype=np.float32)
    target[:-1] = 1.0 / env.n_assets  # 44종목 균등, 현금 0
    action_logit = np.log(np.maximum(target, 1e-12))
    done, values = False, [1.0]
    while not done:
        obs, _, done, _, info = env.step(action_logit)
        values.append(info['portfolio_value'])
    return np.array(values)


SMOOTH_ALPHA_21 = 0.01
ew_valid_cash = equal_weight_cash_backtest(valid_buckets)
ew_test_cash  = equal_weight_cash_backtest(test_buckets)

rows_valid4 = {'등가중 벤치마크': compute_perf(ew_valid_cash)}
rows_test4  = {'등가중 벤치마크': compute_perf(ew_test_cash)}

for beta in POOL_BETAS:
    v_values, _ = run_backtest_smoothed(CashPortfolioEnv, v3_baseline_models[beta], valid_buckets, SMOOTH_ALPHA_21)
    t_values, _ = run_backtest_smoothed(CashPortfolioEnv, v3_baseline_models[beta], test_buckets, SMOOTH_ALPHA_21)
    rows_valid4[f'β={beta} (v3 baseline)'] = compute_perf(v_values)
    rows_test4[f'β={beta} (v3 baseline)']  = compute_perf(t_values)

for beta in POOL_BETAS:
    v_values, _ = run_backtest_smoothed(CashPortfolioEnv, pool_models_objinj_cash[beta], valid_buckets, SMOOTH_ALPHA_21)
    t_values, _ = run_backtest_smoothed(CashPortfolioEnv, pool_models_objinj_cash[beta], test_buckets, SMOOTH_ALPHA_21)
    rows_valid4[f'β={beta} (목적함수 주입)'] = compute_perf(v_values)
    rows_test4[f'β={beta} (목적함수 주입)']  = compute_perf(t_values)

valid_df4 = pd.DataFrame(rows_valid4).T
test_df4  = pd.DataFrame(rows_test4).T
print('--- valid(2025 Q4), 스무딩 적용 ---')
print(valid_df4.to_string())
print('\n--- test(2026~), 스무딩 적용 ---')
print(test_df4.to_string())

valid_df4.to_csv(CRYPTO_DIR / 'crypto_hrl_v3_objinj_valid_smoothed_fix1004.csv', encoding='utf-8-sig')
test_df4.to_csv(CRYPTO_DIR / 'crypto_hrl_v3_objinj_test_smoothed_fix1004.csv', encoding='utf-8-sig')
print('\n저장 완료: crypto_hrl_v3_objinj_valid_smoothed_fix1004.csv, crypto_hrl_v3_objinj_test_smoothed_fix1004.csv')

In [ ]:
print('=== [§21-I] λ=0.15 valid/test 성과 (스무딩 α=0.01 적용) ===\n')

rows_valid5 = {'등가중 벤치마크': compute_perf(ew_valid_cash)}
rows_test5  = {'등가중 벤치마크': compute_perf(ew_test_cash)}

for beta in POOL_BETAS:
    v_values, _ = run_backtest_smoothed(CashPortfolioEnv, pool_models_objinj_cash_strong[beta], valid_buckets, SMOOTH_ALPHA_21)
    t_values, _ = run_backtest_smoothed(CashPortfolioEnv, pool_models_objinj_cash_strong[beta], test_buckets, SMOOTH_ALPHA_21)
    rows_valid5[f'β={beta} (λ=0.15)'] = compute_perf(v_values)
    rows_test5[f'β={beta} (λ=0.15)']  = compute_perf(t_values)

valid_df5 = pd.DataFrame(rows_valid5).T
test_df5  = pd.DataFrame(rows_test5).T
print('--- valid(2025 Q4), λ=0.15, 스무딩 적용 ---')
print(valid_df5.to_string())
print('\n--- test(2026~), λ=0.15, 스무딩 적용 ---')
print(test_df5.to_string())

valid_df5.to_csv(CRYPTO_DIR / 'crypto_hrl_v3_objinj_strong_valid_smoothed_fix1004.csv', encoding='utf-8-sig')
test_df5.to_csv(CRYPTO_DIR / 'crypto_hrl_v3_objinj_strong_test_smoothed_fix1004.csv', encoding='utf-8-sig')
print('\n저장 완료: crypto_hrl_v3_objinj_strong_valid_smoothed_fix1004.csv, crypto_hrl_v3_objinj_strong_test_smoothed_fix1004.csv')

In [ ]:
# 요약 — 논문 4.3절에 옮길 값: β 순서 vs 평균 현금비중 스피어만 ρ, λ=0.15 β별 valid/test 샤프·격차
summary = {}
for label in ['v3 baseline', 'λ=0.01', 'λ=0.15']:
    bs = sorted(cash_means[label].keys())
    summary[label] = {'spearman(β,cash)': pd.Series(bs).corr(pd.Series([cash_means[label][b] for b in bs]), method='spearman'),
                      **{f'cash β={b}': cash_means[label][b] for b in bs}}
print(pd.DataFrame(summary).T.to_string(float_format=lambda x: f'{x:.4f}'))
gap = {b: {'valid': valid_df5.loc[f'β={b} (λ=0.15)', 'sharpe'], 'test': test_df5.loc[f'β={b} (λ=0.15)', 'sharpe']} for b in POOL_BETAS}
gap = pd.DataFrame(gap).T; gap['gap'] = (gap['valid'] - gap['test']).abs()
gap.loc['등가중'] = [valid_df5.loc['등가중 벤치마크', 'sharpe'], test_df5.loc['등가중 벤치마크', 'sharpe'], np.nan]
print(gap.to_string(float_format=lambda x: f'{x:.3f}'))
pd.DataFrame(summary).T.to_csv(CRYPTO_DIR / 'crypto_hrl_v3_objinj_summary_fix1004.csv', encoding='utf-8-sig')
gap.to_csv(CRYPTO_DIR / 'crypto_hrl_v3_objinj_strong_gap_fix1004.csv', encoding='utf-8-sig')
print('저장 완료: crypto_hrl_v3_objinj_summary_fix1004.csv, crypto_hrl_v3_objinj_strong_gap_fix1004.csv')
